# Set up the data pipeline using Snowflake CLI

Snowflake cli have some issue, I used this solution because i suspected snow git features locked .

In [ ]:
%%sql -r dataframe_1
USE ROLE accountadmin;

/*--
database, schema and warehouse creation
--*/


-- create tasty_bytes database
CREATE OR ALTER DATABASE STAGING_tasty_bytes;


-- create raw_pos schema
CREATE OR ALTER SCHEMA STAGING_tasty_bytes.raw_pos;


-- create raw_customer schema
CREATE OR ALTER SCHEMA STAGING_tasty_bytes.raw_customer;


-- create harmonized schema
CREATE OR ALTER SCHEMA STAGING_tasty_bytes.harmonized;


-- create analytics schema
CREATE OR ALTER SCHEMA STAGING_tasty_bytes.analytics;


-- create warehouse for ingestion
CREATE OR REPLACE WAREHOUSE demo_build_wh
   WAREHOUSE_SIZE = 'xlarge'
   WAREHOUSE_TYPE = 'standard'
   AUTO_SUSPEND = 60
   AUTO_RESUME = TRUE
   INITIALLY_SUSPENDED = TRUE;


/*--
file format and stage creation
--*/


CREATE OR ALTER FILE FORMAT STAGING_tasty_bytes.public.csv_ff
type = 'csv';


CREATE OR REPLACE STAGE STAGING_tasty_bytes.public.s3load
url = 's3://sfquickstarts/tasty-bytes-builder-education/'
file_format = STAGING_tasty_bytes.public.csv_ff;


/*--
raw zone table build
--*/


-- country table build

-- todo: complete table build
CREATE TABLE STAGING_tasty_bytes.raw_pos.country
(
   country_id NUMBER(18,0),
   country VARCHAR(16777216),
   iso_currency VARCHAR(3),
   iso_country VARCHAR(2),
   city VARCHAR(16777216),
   city_population VARCHAR(16777216)
);


-- franchise table build
CREATE OR ALTER TABLE STAGING_tasty_bytes.raw_pos.franchise
(
   franchise_id NUMBER(38,0),
   first_name VARCHAR(16777216),
   last_name VARCHAR(16777216),
   city VARCHAR(16777216),
   country VARCHAR(16777216),
   e_mail VARCHAR(16777216),
   phone_number VARCHAR(16777216)
);


-- location table build
CREATE OR ALTER TABLE STAGING_tasty_bytes.raw_pos.location
(
   location_id NUMBER(19,0),
   placekey VARCHAR(16777216),
   location VARCHAR(16777216),
   city VARCHAR(16777216),
   region VARCHAR(16777216),
   iso_country_code VARCHAR(16777216),
   country VARCHAR(16777216)
);


-- menu table build
CREATE OR ALTER TABLE STAGING_tasty_bytes.raw_pos.menu
(
   menu_id NUMBER(19,0),
   menu_type_id NUMBER(38,0),
   menu_type VARCHAR(16777216),
   truck_brand_name VARCHAR(16777216),
   menu_item_id NUMBER(38,0),
   menu_item_name VARCHAR(16777216),
   item_category VARCHAR(16777216),
   item_subcategory VARCHAR(16777216),
   cost_of_goods_usd NUMBER(38,4),
   sale_price_usd NUMBER(38,4),
   menu_item_health_metrics_obj VARIANT
);


-- truck table build
CREATE OR ALTER TABLE STAGING_tasty_bytes.raw_pos.truck
(
   truck_id NUMBER(38,0),
   menu_type_id NUMBER(38,0),
   primary_city VARCHAR(16777216),
   region VARCHAR(16777216),
   iso_region VARCHAR(16777216),
   country VARCHAR(16777216),
   iso_country_code VARCHAR(16777216),
   franchise_flag NUMBER(38,0),
   year NUMBER(38,0),
   make VARCHAR(16777216),
   model VARCHAR(16777216),
   ev_flag NUMBER(38,0),
   franchise_id NUMBER(38,0),
   truck_opening_date DATE
);


-- order_header table build
CREATE OR ALTER TABLE STAGING_tasty_bytes.raw_pos.order_header
(
   order_id NUMBER(38,0),
   truck_id NUMBER(38,0),
   location_id FLOAT,
   customer_id NUMBER(38,0),
   discount_id VARCHAR(16777216),
   shift_id NUMBER(38,0),
   shift_start_time TIME(9),
   shift_end_time TIME(9),
   order_channel VARCHAR(16777216),
   order_ts TIMESTAMP_NTZ(9),
   served_ts VARCHAR(16777216),
   order_currency VARCHAR(3),
   order_amount NUMBER(38,4),
   order_tax_amount VARCHAR(16777216),
   order_discount_amount VARCHAR(16777216),
   order_total NUMBER(38,4)
);


-- order_detail table build
CREATE OR ALTER TABLE STAGING_tasty_bytes.raw_pos.order_detail
(
   order_detail_id NUMBER(38,0),
   order_id NUMBER(38,0),
   menu_item_id NUMBER(38,0),
   discount_id VARCHAR(16777216),
   line_number NUMBER(38,0),
   quantity NUMBER(5,0),
   unit_price NUMBER(38,4),
   price NUMBER(38,4),
   order_item_discount_amount VARCHAR(16777216)
);


-- customer loyalty table build
CREATE OR ALTER TABLE STAGING_tasty_bytes.raw_customer.customer_loyalty
(
   customer_id NUMBER(38,0),
   first_name VARCHAR(16777216),
   last_name VARCHAR(16777216),
   city VARCHAR(16777216),
   country VARCHAR(16777216),
   postal_code VARCHAR(16777216),
   preferred_language VARCHAR(16777216),
   gender VARCHAR(16777216),
   favourite_brand VARCHAR(16777216),
   marital_status VARCHAR(16777216),
   children_count VARCHAR(16777216),
   sign_up_date DATE,
   birthday_date DATE,
   e_mail VARCHAR(16777216),
   phone_number VARCHAR(16777216)
);


/*--
harmonized view creation
--*/


-- orders_v view
CREATE OR REPLACE VIEW STAGING_tasty_bytes.harmonized.orders_v
   AS
SELECT
   oh.order_id,
   oh.truck_id,
   oh.order_ts,
   od.order_detail_id,
   od.line_number,
   m.truck_brand_name,
   m.menu_type,
   t.primary_city,
   t.region,
   t.country,
   t.franchise_flag,
   t.franchise_id,
   f.first_name AS franchisee_first_name,
   f.last_name AS franchisee_last_name,
   l.location_id,
   cl.customer_id,
   cl.first_name,
   cl.last_name,
   cl.e_mail,
   cl.phone_number,
   cl.children_count,
   cl.gender,
   cl.marital_status,
   od.menu_item_id,
   m.menu_item_name,
   od.quantity,
   od.unit_price,
   od.price,
   oh.order_amount,
   oh.order_tax_amount,
   oh.order_discount_amount,
   oh.order_total
FROM STAGING_tasty_bytes.raw_pos.order_detail od
JOIN STAGING_tasty_bytes.raw_pos.order_header oh
   ON od.order_id = oh.order_id
JOIN STAGING_tasty_bytes.raw_pos.truck t
   ON oh.truck_id = t.truck_id
JOIN STAGING_tasty_bytes.raw_pos.menu m
   ON od.menu_item_id = m.menu_item_id
JOIN STAGING_tasty_bytes.raw_pos.franchise f
   ON t.franchise_id = f.franchise_id
JOIN STAGING_tasty_bytes.raw_pos.location l
   ON oh.location_id = l.location_id
LEFT JOIN STAGING_tasty_bytes.raw_customer.customer_loyalty cl
   ON oh.customer_id = cl.customer_id;


-- loyalty_metrics_v view
CREATE OR REPLACE VIEW STAGING_tasty_bytes.harmonized.customer_loyalty_metrics_v
   AS
SELECT
   cl.customer_id,
   cl.city,
   cl.country,
   cl.first_name,
   cl.last_name,
   cl.phone_number,
   cl.e_mail,
   SUM(oh.order_total) AS total_sales,
   ARRAY_AGG(DISTINCT oh.location_id) AS visited_location_ids_array
FROM STAGING_tasty_bytes.raw_customer.customer_loyalty cl
JOIN STAGING_tasty_bytes.raw_pos.order_header oh
ON cl.customer_id = oh.customer_id
GROUP BY cl.customer_id, cl.city, cl.country, cl.first_name,
cl.last_name, cl.phone_number, cl.e_mail;


/*--
analytics view creation
--*/


-- orders_v view
CREATE OR REPLACE VIEW STAGING_tasty_bytes.analytics.orders_v
COMMENT = 'Tasty Bytes Order Detail View'
   AS
SELECT DATE(o.order_ts) AS date, * FROM STAGING_tasty_bytes.harmonized.orders_v o;


-- customer_loyalty_metrics_v view
CREATE OR REPLACE VIEW STAGING_tasty_bytes.analytics.customer_loyalty_metrics_v
COMMENT = 'Tasty Bytes Customer Loyalty Member Metrics View'
   AS
SELECT * FROM STAGING_tasty_bytes.harmonized.customer_loyalty_metrics_v;


/*--
raw zone table load
--*/


USE WAREHOUSE demo_build_wh;


-- country table load
-- COPY INTO STAGING_tasty_bytes.raw_pos.country
-- (
--    country_id,
--    country,
--    iso_currency,
--    iso_country,
--    city_id,
--    city,
--    city_population
-- )
-- FROM @STAGING_tasty_bytes.public.s3load/raw_pos/country/;


-- franchise table load
COPY INTO STAGING_tasty_bytes.raw_pos.franchise
FROM @STAGING_tasty_bytes.public.s3load/raw_pos/franchise/;


-- location table load
COPY INTO STAGING_tasty_bytes.raw_pos.location
FROM @STAGING_tasty_bytes.public.s3load/raw_pos/location/;


-- menu table load
COPY INTO STAGING_tasty_bytes.raw_pos.menu
FROM @STAGING_tasty_bytes.public.s3load/raw_pos/menu/;


-- truck table load
COPY INTO STAGING_tasty_bytes.raw_pos.truck
FROM @STAGING_tasty_bytes.public.s3load/raw_pos/truck/;


-- customer_loyalty table load
COPY INTO STAGING_tasty_bytes.raw_customer.customer_loyalty
FROM @STAGING_tasty_bytes.public.s3load/raw_customer/customer_loyalty/;


-- order_header table load
COPY INTO STAGING_tasty_bytes.raw_pos.order_header
FROM @STAGING_tasty_bytes.public.s3load/raw_pos/subset_order_header/;


-- order_detail table load
COPY INTO STAGING_tasty_bytes.raw_pos.order_detail
FROM @STAGING_tasty_bytes.public.s3load/raw_pos/subset_order_detail/;

In [ ]:
%%sql -r dataframe_2
USE ROLE accountadmin;

/*--
database, schema and warehouse creation
--*/


-- create tasty_bytes database
CREATE OR ALTER DATABASE PROD_tasty_bytes;


-- create raw_pos schema
CREATE OR ALTER SCHEMA PROD_tasty_bytes.raw_pos;


-- create raw_customer schema
CREATE OR ALTER SCHEMA PROD_tasty_bytes.raw_customer;


-- create harmonized schema
CREATE OR ALTER SCHEMA PROD_tasty_bytes.harmonized;


-- create analytics schema
CREATE OR ALTER SCHEMA PROD_tasty_bytes.analytics;


-- create warehouse for ingestion
CREATE OR REPLACE WAREHOUSE demo_build_wh
   WAREHOUSE_SIZE = 'xlarge'
   WAREHOUSE_TYPE = 'standard'
   AUTO_SUSPEND = 60
   AUTO_RESUME = TRUE
   INITIALLY_SUSPENDED = TRUE;


/*--
file format and stage creation
--*/


CREATE OR ALTER FILE FORMAT PROD_tasty_bytes.public.csv_ff
type = 'csv';


CREATE OR REPLACE STAGE PROD_tasty_bytes.public.s3load
url = 's3://sfquickstarts/tasty-bytes-builder-education/'
file_format = PROD_tasty_bytes.public.csv_ff;


/*--
raw zone table build
--*/


-- country table build

-- todo: complete table build
CREATE TABLE PROD_tasty_bytes.raw_pos.country
(
   country_id NUMBER(18,0),
   country VARCHAR(16777216),
   iso_currency VARCHAR(3),
   iso_country VARCHAR(2),
   city VARCHAR(16777216),
   city_population VARCHAR(16777216)
);


-- franchise table build
CREATE OR ALTER TABLE PROD_tasty_bytes.raw_pos.franchise
(
   franchise_id NUMBER(38,0),
   first_name VARCHAR(16777216),
   last_name VARCHAR(16777216),
   city VARCHAR(16777216),
   country VARCHAR(16777216),
   e_mail VARCHAR(16777216),
   phone_number VARCHAR(16777216)
);


-- location table build
CREATE OR ALTER TABLE PROD_tasty_bytes.raw_pos.location
(
   location_id NUMBER(19,0),
   placekey VARCHAR(16777216),
   location VARCHAR(16777216),
   city VARCHAR(16777216),
   region VARCHAR(16777216),
   iso_country_code VARCHAR(16777216),
   country VARCHAR(16777216)
);


-- menu table build
CREATE OR ALTER TABLE PROD_tasty_bytes.raw_pos.menu
(
   menu_id NUMBER(19,0),
   menu_type_id NUMBER(38,0),
   menu_type VARCHAR(16777216),
   truck_brand_name VARCHAR(16777216),
   menu_item_id NUMBER(38,0),
   menu_item_name VARCHAR(16777216),
   item_category VARCHAR(16777216),
   item_subcategory VARCHAR(16777216),
   cost_of_goods_usd NUMBER(38,4),
   sale_price_usd NUMBER(38,4),
   menu_item_health_metrics_obj VARIANT
);


-- truck table build
CREATE OR ALTER TABLE PROD_tasty_bytes.raw_pos.truck
(
   truck_id NUMBER(38,0),
   menu_type_id NUMBER(38,0),
   primary_city VARCHAR(16777216),
   region VARCHAR(16777216),
   iso_region VARCHAR(16777216),
   country VARCHAR(16777216),
   iso_country_code VARCHAR(16777216),
   franchise_flag NUMBER(38,0),
   year NUMBER(38,0),
   make VARCHAR(16777216),
   model VARCHAR(16777216),
   ev_flag NUMBER(38,0),
   franchise_id NUMBER(38,0),
   truck_opening_date DATE
);


-- order_header table build
CREATE OR ALTER TABLE PROD_tasty_bytes.raw_pos.order_header
(
   order_id NUMBER(38,0),
   truck_id NUMBER(38,0),
   location_id FLOAT,
   customer_id NUMBER(38,0),
   discount_id VARCHAR(16777216),
   shift_id NUMBER(38,0),
   shift_start_time TIME(9),
   shift_end_time TIME(9),
   order_channel VARCHAR(16777216),
   order_ts TIMESTAMP_NTZ(9),
   served_ts VARCHAR(16777216),
   order_currency VARCHAR(3),
   order_amount NUMBER(38,4),
   order_tax_amount VARCHAR(16777216),
   order_discount_amount VARCHAR(16777216),
   order_total NUMBER(38,4)
);


-- order_detail table build
CREATE OR ALTER TABLE PROD_tasty_bytes.raw_pos.order_detail
(
   order_detail_id NUMBER(38,0),
   order_id NUMBER(38,0),
   menu_item_id NUMBER(38,0),
   discount_id VARCHAR(16777216),
   line_number NUMBER(38,0),
   quantity NUMBER(5,0),
   unit_price NUMBER(38,4),
   price NUMBER(38,4),
   order_item_discount_amount VARCHAR(16777216)
);


-- customer loyalty table build
CREATE OR ALTER TABLE PROD_tasty_bytes.raw_customer.customer_loyalty
(
   customer_id NUMBER(38,0),
   first_name VARCHAR(16777216),
   last_name VARCHAR(16777216),
   city VARCHAR(16777216),
   country VARCHAR(16777216),
   postal_code VARCHAR(16777216),
   preferred_language VARCHAR(16777216),
   gender VARCHAR(16777216),
   favourite_brand VARCHAR(16777216),
   marital_status VARCHAR(16777216),
   children_count VARCHAR(16777216),
   sign_up_date DATE,
   birthday_date DATE,
   e_mail VARCHAR(16777216),
   phone_number VARCHAR(16777216)
);


/*--
harmonized view creation
--*/


-- orders_v view
CREATE OR REPLACE VIEW PROD_tasty_bytes.harmonized.orders_v
   AS
SELECT
   oh.order_id,
   oh.truck_id,
   oh.order_ts,
   od.order_detail_id,
   od.line_number,
   m.truck_brand_name,
   m.menu_type,
   t.primary_city,
   t.region,
   t.country,
   t.franchise_flag,
   t.franchise_id,
   f.first_name AS franchisee_first_name,
   f.last_name AS franchisee_last_name,
   l.location_id,
   cl.customer_id,
   cl.first_name,
   cl.last_name,
   cl.e_mail,
   cl.phone_number,
   cl.children_count,
   cl.gender,
   cl.marital_status,
   od.menu_item_id,
   m.menu_item_name,
   od.quantity,
   od.unit_price,
   od.price,
   oh.order_amount,
   oh.order_tax_amount,
   oh.order_discount_amount,
   oh.order_total
FROM PROD_tasty_bytes.raw_pos.order_detail od
JOIN PROD_tasty_bytes.raw_pos.order_header oh
   ON od.order_id = oh.order_id
JOIN PROD_tasty_bytes.raw_pos.truck t
   ON oh.truck_id = t.truck_id
JOIN PROD_tasty_bytes.raw_pos.menu m
   ON od.menu_item_id = m.menu_item_id
JOIN PROD_tasty_bytes.raw_pos.franchise f
   ON t.franchise_id = f.franchise_id
JOIN PROD_tasty_bytes.raw_pos.location l
   ON oh.location_id = l.location_id
LEFT JOIN PROD_tasty_bytes.raw_customer.customer_loyalty cl
   ON oh.customer_id = cl.customer_id;


-- loyalty_metrics_v view
CREATE OR REPLACE VIEW PROD_tasty_bytes.harmonized.customer_loyalty_metrics_v
   AS
SELECT
   cl.customer_id,
   cl.city,
   cl.country,
   cl.first_name,
   cl.last_name,
   cl.phone_number,
   cl.e_mail,
   SUM(oh.order_total) AS total_sales,
   ARRAY_AGG(DISTINCT oh.location_id) AS visited_location_ids_array
FROM PROD_tasty_bytes.raw_customer.customer_loyalty cl
JOIN PROD_tasty_bytes.raw_pos.order_header oh
ON cl.customer_id = oh.customer_id
GROUP BY cl.customer_id, cl.city, cl.country, cl.first_name,
cl.last_name, cl.phone_number, cl.e_mail;


/*--
analytics view creation
--*/


-- orders_v view
CREATE OR REPLACE VIEW PROD_tasty_bytes.analytics.orders_v
COMMENT = 'Tasty Bytes Order Detail View'
   AS
SELECT DATE(o.order_ts) AS date, * FROM PROD_tasty_bytes.harmonized.orders_v o;


-- customer_loyalty_metrics_v view
CREATE OR REPLACE VIEW PROD_tasty_bytes.analytics.customer_loyalty_metrics_v
COMMENT = 'Tasty Bytes Customer Loyalty Member Metrics View'
   AS
SELECT * FROM PROD_tasty_bytes.harmonized.customer_loyalty_metrics_v;


/*--
raw zone table load
--*/


USE WAREHOUSE demo_build_wh;


-- country table load
-- COPY INTO PROD_tasty_bytes.raw_pos.country
-- (
--    country_id,
--    country,
--    iso_currency,
--    iso_country,
--    city_id,
--    city,
--    city_population
-- )
-- FROM @PROD_tasty_bytes.public.s3load/raw_pos/country/;


-- franchise table load
COPY INTO PROD_tasty_bytes.raw_pos.franchise
FROM @PROD_tasty_bytes.public.s3load/raw_pos/franchise/;


-- location table load
COPY INTO PROD_tasty_bytes.raw_pos.location
FROM @PROD_tasty_bytes.public.s3load/raw_pos/location/;


-- menu table load
COPY INTO PROD_tasty_bytes.raw_pos.menu
FROM @PROD_tasty_bytes.public.s3load/raw_pos/menu/;


-- truck table load
COPY INTO PROD_tasty_bytes.raw_pos.truck
FROM @PROD_tasty_bytes.public.s3load/raw_pos/truck/;


-- customer_loyalty table load
COPY INTO PROD_tasty_bytes.raw_customer.customer_loyalty
FROM @PROD_tasty_bytes.public.s3load/raw_customer/customer_loyalty/;


-- order_header table load
COPY INTO PROD_tasty_bytes.raw_pos.order_header
FROM @PROD_tasty_bytes.public.s3load/raw_pos/subset_order_header/;


-- order_detail table load
COPY INTO PROD_tasty_bytes.raw_pos.order_detail
FROM @PROD_tasty_bytes.public.s3load/raw_pos/subset_order_detail/;

In [ ]:
%%sql -r dataframe_3
/*-- database/tasty_bytes.sql --*/
CREATE OR ALTER DATABASE STAGING_tasty_bytes;

/*-- functions/fahrenheit_to_celsius.sql --*/
CREATE OR ALTER FUNCTION STAGING_tasty_bytes.analytics.fahrenheit_to_celsius(temp_f NUMBER(35,4))
  RETURNS NUMBER(35,4)
  AS
  $$
    (temp_f - 32) * (5/9)
  $$
;

/*-- functions/inch_to_milimeter.sql --*/
CREATE OR ALTER FUNCTION STAGING_tasty_bytes.analytics.inch_to_millimeter(inch NUMBER(35,4))
  RETURNS NUMBER(35,4)
  AS
  $$
    inch * 25.4
  $$
;

/*-- sprocs/process_order_headers_stream.sql --*/
-- Create the stored procedure, define its logic with Snowpark for Python, write sales to raw_pos.daily_sales_hamburg_t
CREATE OR REPLACE PROCEDURE STAGING_tasty_bytes.raw_pos.process_order_headers_stream()
  RETURNS STRING
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.10'
  HANDLER ='process_order_headers_stream'
  PACKAGES = ('snowflake-snowpark-python')
AS
$$
import snowflake.snowpark.functions as F
from snowflake.snowpark import Session

def process_order_headers_stream(session: Session) -> float:
    # Query the stream
    recent_orders = session.table("order_header_stream").filter(F.col("METADATA$ACTION") == "INSERT")
    
    # Look up location of the orders in the stream using the LOCATIONS table
    locations = session.table("location")
    hamburg_orders = recent_orders.join(
        locations,
        recent_orders["LOCATION_ID"] == locations["LOCATION_ID"]
    ).filter(
        (locations["CITY"] == "Hamburg") &
        (locations["COUNTRY"] == "Germany")
    )
    
    # Calculate the sum of sales in Hamburg
    total_sales = hamburg_orders.group_by(F.date_trunc('DAY', F.col("ORDER_TS"))).agg(
        F.coalesce(F.sum("ORDER_TOTAL"), F.lit(0)).alias("total_sales")
    )
    
    # Select the columns with proper aliases and convert to date type
    daily_sales = total_sales.select(
        F.col("DATE_TRUNC('DAY', ORDER_TS)").cast("DATE").alias("DATE"),
        F.col("total_sales")
    )
    
    # Write the results to the DAILY_SALES_HAMBURG_T table
    total_sales.write.mode("append").save_as_table("raw_pos.daily_sales_hamburg_t")
    
    # Return a message indicating the operation was successful
    return "Daily sales for Hamburg, Germany have been successfully written to raw_pos.daily_sales_hamburg_t"
$$;

/*-- streams/order_header.sql --*/

CREATE OR REPLACE STREAM STAGING_tasty_bytes.raw_pos.order_header_stream ON TABLE STAGING_tasty_bytes.raw_pos.order_header;

/*-- tables/daily_sales_hamburg.sql --*/
CREATE OR REPLACE DYNAMIC TABLE STAGING_tasty_bytes.raw_pos.daily_sales_hamburg
WAREHOUSE = 'COMPUTE_WH'
TARGET_LAG = '1 minute'
AS
SELECT
    CAST(oh.ORDER_TS AS DATE) AS date,
    COALESCE(SUM(oh.ORDER_TOTAL), 0) AS total_sales
FROM
    STAGING_tasty_bytes.raw_pos.order_header oh
JOIN
    STAGING_tasty_bytes.raw_pos.location loc
ON
    oh.LOCATION_ID = loc.LOCATION_ID
WHERE
    loc.CITY = 'Hamburg'
    AND loc.COUNTRY = 'Germany'
GROUP BY
    CAST(oh.ORDER_TS AS DATE);

/*-- views/views.sql --*/
-- Create view that adds weather data for cities where Tasty Bytes operates
CREATE OR REPLACE VIEW STAGING_tasty_bytes.harmonized.daily_weather_v
COMMENT = 'Weather Source Daily History filtered to Tasty Bytes supported Cities'
    AS
SELECT
    hd.*,
    TO_VARCHAR(hd.date_valid_std, 'YYYY-MM') AS yyyy_mm,
    pc.city_name AS city,
    c.country AS country_desc
FROM PELMOREX_WEATHER_SOURCE_FROSTBYTE.onpoint_id.history_day hd
JOIN PELMOREX_WEATHER_SOURCE_FROSTBYTE.onpoint_id.postal_codes pc
    ON pc.postal_code = hd.postal_code
    AND pc.country = hd.country
JOIN STAGING_tasty_bytes.raw_pos.country c
    ON c.iso_country = hd.country
    AND c.city = hd.city_name;

-- Apply UDFs and confirm successful execution
CREATE OR REPLACE VIEW STAGING_tasty_bytes.harmonized.weather_hamburg
AS
SELECT
    fd.date_valid_std AS date,
    fd.city_name,
    fd.country_desc,
    ZEROIFNULL(SUM(odv.price)) AS daily_sales,
    ROUND(AVG(fd.avg_temperature_air_2m_f),2) AS avg_temperature_fahrenheit,
    ROUND(AVG(analytics.fahrenheit_to_celsius(fd.avg_temperature_air_2m_f)),2) AS avg_temperature_celsius,
    ROUND(AVG(fd.tot_precipitation_in),2) AS avg_precipitation_inches,
    ROUND(AVG(analytics.inch_to_millimeter(fd.tot_precipitation_in)),2) AS avg_precipitation_millimeters,
    MAX(fd.max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM harmonized.daily_weather_v fd
LEFT JOIN harmonized.orders_v odv
    ON fd.date_valid_std = DATE(odv.order_ts)
    AND fd.city_name = odv.primary_city
    AND fd.country_desc = odv.country
WHERE 1=1
    AND fd.country_desc = 'Germany'
    AND fd.city = 'Hamburg'
    AND fd.yyyy_mm = '2022-02'
GROUP BY fd.date_valid_std, fd.city_name, fd.country_desc
ORDER BY fd.date_valid_std ASC;

-- Expand tracking to all cities and deploy view with this new information
CREATE OR REPLACE VIEW STAGING_tasty_bytes.analytics.daily_city_metrics_v
COMMENT = 'Daily Weather Metrics and Orders Data'
AS
SELECT
    fd.date_valid_std AS date,
    fd.city_name,
    fd.country_desc,
    ZEROIFNULL(SUM(odv.price)) AS daily_sales,
    ROUND(AVG(fd.avg_temperature_air_2m_f),2) AS avg_temperature_fahrenheit,
    ROUND(AVG(STAGING_tasty_bytes.analytics.fahrenheit_to_celsius(fd.avg_temperature_air_2m_f)),2) AS avg_temperature_celsius,
    ROUND(AVG(fd.tot_precipitation_in),2) AS avg_precipitation_inches,
    ROUND(AVG(STAGING_tasty_bytes.analytics.inch_to_millimeter(fd.tot_precipitation_in)),2) AS avg_precipitation_millimeters,
    MAX(fd.max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM STAGING_tasty_bytes.harmonized.daily_weather_v fd
LEFT JOIN STAGING_tasty_bytes.harmonized.orders_v odv
    ON fd.date_valid_std = DATE(odv.order_ts)
    AND fd.city_name = odv.primary_city
    AND fd.country_desc = odv.country
GROUP BY fd.date_valid_std, fd.city_name, fd.country_desc;

-- Create a view that tracks windspeed for Hamburg, Germany
CREATE OR REPLACE VIEW STAGING_tasty_bytes.harmonized.windspeed_hamburg
    AS
SELECT
    dw.country_desc,
    dw.city_name,
    dw.date_valid_std,
    MAX(dw.max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM harmonized.daily_weather_v dw
WHERE 1=1
    AND dw.country_desc IN ('Germany')
    AND dw.city_name = 'Hamburg'
GROUP BY dw.country_desc, dw.city_name, dw.date_valid_std
ORDER BY dw.date_valid_std DESC;